# 03 — results

**10/10 relations hold.** Three of them held only after correcting my own
assertions; those corrections are the most useful part of this notebook and are
recorded first, not last.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd()))
from engine.e_collatz_shift import (
    T, parity_word, parity_vector, branch_affine, rational_cycle,
    CollatzShiftEngine, run,
)

In [2]:
result = run(verbose=False)
for name, tier, status, prov, claim in result['relations']:
    print(f'{name:<30} t{tier}  {status:<11} {prov}')

collatz.affine_on_classes      t1  HOLDS       KNOWN (folklore)
collatz.parity_bijection       t3  HOLDS       KNOWN (Bernstein 1994; Bernstein-Lagarias 1996)
collatz.shift_conjugacy        t2  HOLDS       KNOWN (Bernstein-Lagarias 1996)
collatz.pascal_refinement      t3  HOLDS       OURS (framing; the maths is elementary once stated)
collatz.gaussian_from_pascal   t3  HOLDS       OURS (framing)
collatz.cycle_denominator      t3  HOLDS       KNOWN (standard; 2^k-3^d=1 only at (2,1) is Levi ben Gerson / Catalan, cited not proved here)
collatz.mod3_orphans           t2  HOLDS       KNOWN (elementary)
collatz.forward_two_to_one     t3  HOLDS       OURS (the measurement; the fact follows from R3)
collatz.backward_deficit       t3  HOLDS       OURS (the measurement)
collatz.drift_at_half          t3  HOLDS       KNOWN (Terras 1976; Everett 1977)


## The three faults on the first run — all mine, none the mathematics

| relation | what the harness said | what was actually wrong |
|---|---|---|
| `collatz.cycle_denominator` | MATHS-FAULT | My necklace filter kept rotation-minimal words but not **primitive** ones, so `1010`, `101010`, … came through and the single loop `{1,2}` was reported as seven distinct cycles. Fixed: primitive necklaces only. Now exactly one positive-integer cycle to `k=16`. |
| `collatz.forward_two_to_one` | MATHS-FAULT | The **measurement was correct** (fibre size 2) and my assertion expected 4. A CODE fault dressed as a maths fault. |
| `collatz.backward_deficit` | MATHS-FAULT | I predicted mean out-degree `7/6` from a condition that does not exist. |

### The third one made the result better

I had written the odd-predecessor test as *"`n ≡ 2 (mod 3)` **and** `(2n−1)/3` is
odd"*. The second half is **vacuous**: `2n−1` is odd and `3` is odd, so the
quotient is odd whenever it is an integer at all. The engine now measures that
vacuity rather than assuming it.

With the phantom condition removed the density is exactly `1/3`, the mean
backward out-degree is exactly `4/3`, and the deficit against the forward
direction comes out **exact**:

In [3]:
import math
print(f'forward  destroys  log 2     = {math.log(2):.6f} nats/step  (T is 2-to-1, R8)')
print(f'backward restores  log(4/3)  = {math.log(4/3):.6f} nats/step  (mean out-degree, R9)')
print(f'shortfall                    = {math.log(2)-math.log(4/3):.6f}')
print(f'log(3/2)                     = {math.log(1.5):.6f}')
print(f'exact?  {abs((math.log(2)-math.log(4/3)) - math.log(1.5)) < 1e-15}')

forward  destroys  log 2     = 0.693147 nats/step  (T is 2-to-1, R8)
backward restores  log(4/3)  = 0.287682 nats/step  (mean out-degree, R9)
shortfall                    = 0.405465
log(3/2)                     = 0.405465
exact?  True


## The result, in one paragraph

The Collatz map is the one-sided binary shift on `Z_2`, under an explicit
measure-preserving change of coordinates (the parity vector). Its piecewise
structure is affine on residue classes mod `2^k`, its branch tree is Pascal's
triangle exactly, and its cycles are rationals with denominator `2^k − 3^d`. The
dynamics are therefore completely understood and completely chaotic. The
conjecture asserts that `N` — dense, measure zero — misses all of that chaos. It
is not a statement about the map; it is a statement about how two metrics, the
2-adic and the archimedean, fail to see each other.

Decomposed against the operation domain, `T = ADD ∘ SCALE ∘ SIGN`: the smallest
non-trivial composition using all three tier-0 irreducibles at once. **No new
generator is required**, and the `+1` — the sole coupling between the two axes,
which have different identities — is the whole of the difficulty.

## The direction result

| direction | what happens | rate |
|---|---|---|
| **down** (forward `T`) | exactly 2-to-1; one parity bit destroyed per step; provenance discarded | `log 2` nats/step |
| **up** (backward tree on `N`) | mean out-degree `4/3`; enumerates possibilities, propagates nothing | `log(4/3)` nats/step |
| **shortfall** | | `log(3/2)`, exactly |

Going up the tower does not propagate information — it enumerates. Going down
does, and what propagates is the address. The gap between the two directions is
`log(3/2)`, the odd branch's own gain: **the arithmetic the 2-adic metric cannot
see, priced per step.**

## Open

- **The whole conjecture.** Nothing here bears on it. Stated plainly so it cannot
  be read otherwise.
- **Two kinds of orphan.** `sieve_clock.py` models the temporary kind (adopted at
  `2N`). Collatz supplies a clean permanent kind. What distinguishes them is not
  modelled.
- **Is `log(3/2)` per step the right price?** It is exact for this map. Whether
  the same construction on `(pn+1)/q` gives `log(p/q)` is not tested here.